# YouTube Transcript RAG

A simple Retrieval-Augmented Generation pipeline using semantic search over a YouTube video transcript.

The pipeline:

1. Extract the transcript from a YouTube video
2. Split the transcript into meaningful chunks
3. Generate embeddings for the chunks (OpenAI embeddings)
4. Store the embeddings in a vector database (Qdrant)
5. Accept a user query and retrieve the most relevant chunks
6. Generate an answer using the retrieved context

## Setup and Dependencies

In [2]:
# Install the necessary libraries
!pip install openai
!pip install qdrant_client
!pip install youtube-transcript-api

'pip' is not recognized as an internal or external command,
operable program or batch file.


'pip' is not recognized as an internal or external command,
operable program or batch file.
'pip' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
# Import basic libraries
import json                 # For parsing and structuring JSON data

# Credentials — Colab Secrets when on Colab, a local .env otherwise
try:
    from google.colab import userdata          # Colab: keys live in the 🔑 Secrets panel
    IN_COLAB = True
except ImportError:                            # Local Jupyter: keys live in .env
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    IN_COLAB = False

    class userdata:                            # same .get() call works in both places
        @staticmethod
        def get(name):
            import os
            return os.getenv(name) or os.getenv(name.lower())

# OS operations
import os                   # Useful for accessing environment variables and managing paths

# OpenAI API client
from openai import OpenAI   # Official OpenAI client library, used for embeddings and answer generation

# Text processing
import re                   # Regular expressions for cleaning or preprocessing inputs (if needed)

# Vector database client
from qdrant_client import models

import qdrant_client

In [4]:
# Securely retrieve the OpenAI API key (Colab Secrets or local .env)
# This avoids hardcoding sensitive credentials directly in the notebook
openai_api_key = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client with the retrieved API key
# This client will be used for:
# - Generating embeddings for transcript chunks and queries
# - Generating answers from retrieved context
openaiclient = OpenAI(api_key=openai_api_key)

## 1. Extract the Transcript

We pull the transcript of the chosen YouTube video and keep it in `transcript_text`, the variable the rest of the pipeline works with.

The transcript is also saved to `TRANSCRIPT.md`, with the video title as its first line. On later runs the cell checks that file first: if it exists, has content, and its title matches the video in `VIDEO_URL`, the saved transcript is reused and nothing is extracted again. Changing `VIDEO_URL` to a different video triggers a fresh extraction.

In [5]:
import urllib.parse
import urllib.request

from youtube_transcript_api import YouTubeTranscriptApi

# The video used as the knowledge source — change this link to use another video
VIDEO_URL = "https://www.youtube.com/watch?v=4mqjv9swFwE"

# Local cache of the transcript: first line is "# <video title>", the rest is the transcript text
TRANSCRIPT_PATH = "TRANSCRIPT.md"


def get_video_id(video_url: str) -> str:
    """Extracts the 11-character video id from a YouTube link (watch, youtu.be, embed or shorts)."""
    match = re.search(r"(?:v=|youtu\.be/|embed/|shorts/)([\w-]{11})", video_url)
    if not match:
        raise ValueError(f"Could not find a video id in: {video_url}")
    return match.group(1)


def get_video_title(video_url: str) -> str:
    """Looks up the video title through YouTube's public oEmbed endpoint (no API key needed)."""
    oembed_url = "https://www.youtube.com/oembed?" + urllib.parse.urlencode(
        {"url": video_url, "format": "json"})
    with urllib.request.urlopen(oembed_url) as response:
        return json.load(response)["title"].strip()


def load_cached_transcript(path: str, expected_title: str):
    """
    Returns the transcript saved in `path`, or None when it has to be extracted again:
    the file is missing, it is empty, or its title belongs to a different video.
    """
    if not os.path.exists(path):
        return None

    with open(path, encoding="utf-8") as f:
        title_line, _, body = f.read().partition("\n")

    if title_line.strip() != f"# {expected_title}" or not body.strip():
        return None
    return body.strip()


def fetch_transcript(video_id: str) -> str:
    """Downloads the transcript from YouTube and joins its snippets into a single text."""
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
    return " ".join(snippet.text.replace("\n", " ").strip() for snippet in transcript)


video_title = get_video_title(VIDEO_URL)

# transcript_text is the variable passed down the pipeline
transcript_text = load_cached_transcript(TRANSCRIPT_PATH, video_title)

if transcript_text:
    print(f"Using the transcript already saved in {TRANSCRIPT_PATH}")
else:
    print("Extracting the transcript from YouTube ...")
    transcript_text = fetch_transcript(get_video_id(VIDEO_URL))
    with open(TRANSCRIPT_PATH, "w", encoding="utf-8") as f:
        f.write(f"# {video_title}\n\n{transcript_text}\n")
    print(f"Transcript saved to {TRANSCRIPT_PATH}")

print(f"Title: {video_title}")
print(f"Transcript length: {len(transcript_text):,} characters")
print(transcript_text[:300] + " ...")

Extracting the transcript from YouTube ...
Transcript saved to TRANSCRIPT.md
Title: WATCH THIS: DevOps  vs MLOps vs LLMOps explained right !
Transcript length: 6,709 characters
Hi everyone, I hope you're doing well. In this video, I will show you the difference between DevOps, MLOps, and LLMOPS. So to give you an idea about what each term means, for DevOps, it's actually the combination between two words, development and operations. For MLOps, it's actually the same term D ...


## 2. Split the Transcript into Chunks

We use naive fixed-size chunking: a window of `CHUNK_SIZE` characters slides over the transcript, and each chunk repeats the last `CHUNK_OVERLAP_PCT` % of the previous one so that an idea cut at a chunk boundary still appears whole in one of the two chunks.

The result is `chunks`, a list of dicts. Each one carries the chunk `text` (what we will embed), an `id` (the Qdrant point id) and its `start_char` / `end_char` position in the transcript (stored as payload next to the vector).

In [6]:
# Chunking controls
CHUNK_SIZE = 1000          # number of characters per chunk
CHUNK_OVERLAP_PCT = 25     # overlap between consecutive chunks, as a % of CHUNK_SIZE


def chunk_text(text: str, chunk_size: int, overlap_pct: float) -> list:
    """
    Naive fixed-size chunking: slides a window of `chunk_size` characters over the text,
    each window starting `overlap_pct` % of a chunk before the previous one ended.

    Returns a list of dicts, one per chunk:
        id          position of the chunk in the transcript (used as the Qdrant point id)
        text        the chunk itself (what gets embedded)
        start_char  where the chunk starts in the transcript
        end_char    where the chunk ends in the transcript
    """
    if not 0 <= overlap_pct < 100:
        raise ValueError("overlap_pct must be between 0 and 100 (100 excluded)")

    overlap = int(chunk_size * overlap_pct / 100)
    step = chunk_size - overlap

    chunks = []
    for start in range(0, len(text), step):
        end = min(start + chunk_size, len(text))
        chunks.append({
            "id": len(chunks),
            "text": text[start:end],
            "start_char": start,
            "end_char": end,
        })
        if end == len(text):   # the last window reached the end of the transcript
            break
    return chunks


# chunks is the structure passed down the pipeline (embeddings -> Qdrant)
chunks = chunk_text(transcript_text, CHUNK_SIZE, CHUNK_OVERLAP_PCT)

print(f"Chunk size: {CHUNK_SIZE} characters | overlap: {CHUNK_OVERLAP_PCT}%")
print(f"Number of chunks: {len(chunks)}")
print(json.dumps(chunks[1], indent=2))

Chunk size: 1000 characters | overlap: 25%
Number of chunks: 9
{
  "id": 1,
  "text": "rprone and insecure. There is a whole life cycle behind deops that you see here. It starts with planning which consists on planning for tasks. This is followed by coding. So developing code in collaboration followed by building applications. Running tests to validate this code. Release a new version once the code has been validated and ready for deployment. Once deployment is done, operations steps take place which is related to applications infrastructure including scaling when needed, security, data management and backup and recovery. Once the infrastructure is set up, we need to monitor our application by tracking its performance and errors, identify issues or areas for improvement. We talked about deployment but to ensure this we generally need three main environments. The first one development environment which is for developers to develop the code and the application. We have the staging enviro

## 3. Generate Embeddings for the Chunks

Each chunk is converted into a dense vector with OpenAI's `text-embedding-3-small` model. These vectors capture the meaning of the text, so chunks about similar ideas end up close to each other.

`get_text_embeddings()` accepts a list of texts (all chunks are embedded in a single API call) or a single string, we will reuse it later to embed the user query with the same model. The result is `chunk_embeddings`, where `chunk_embeddings[i]` is the vector for `chunks[i]`.

In [7]:
# OpenAI embedding model used for both the transcript chunks and, later, the user queries
EMBEDDING_MODEL = "text-embedding-3-small"


def get_text_embeddings(texts):
    """
    Converts text into dense embeddings using the OpenAI embedding model.
    These embeddings are stored in Qdrant and used to find semantically relevant chunks.

    Args:
        texts (str | list[str]): A single text (e.g. a user query) or a list of texts (e.g. the chunks).

    Returns:
        list[float] for a single text, or list[list[float]] (same order as the input) for a list.
    """
    single = isinstance(texts, str)
    response = openaiclient.embeddings.create(
        model=EMBEDDING_MODEL,
        input=[texts] if single else texts,
    )
    vectors = [item.embedding for item in response.data]
    return vectors[0] if single else vectors


# chunk_embeddings[i] is the vector for chunks[i]
chunk_embeddings = get_text_embeddings([chunk["text"] for chunk in chunks])

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Embedded {len(chunk_embeddings)} chunks, {len(chunk_embeddings[0])} dimensions each")
print(chunk_embeddings[0][:5])  # Print first 5 dimensions for inspection

Embedding model: text-embedding-3-small
Embedded 9 chunks, 1536 dimensions each
[0.020477294921875, 0.033203125, 0.0231170654296875, -0.034942626953125, 0.06524658203125]


## 4. Store the Embeddings in Qdrant

We store the vectors in a local Qdrant database, kept in a `qdrant_data` folder next to the notebook — no server needed.

Each chunk becomes one point in the `youtube_transcript` collection: its embedding is the vector, and the chunk itself (`text`, `id`, `start_char`, `end_char`) is the payload returned at retrieval time. The collection is rebuilt on every run so it always matches the current video and chunking settings.

In [8]:
# 🗄️ Initializing Qdrant client with the local path to the vector database
# The data sits in a qdrant_data folder next to the notebook (or in /content on Colab)
QDRANT_PATH = os.path.join(os.getcwd(), "qdrant_data")
COLLECTION_NAME = "youtube_transcript"

# Local Qdrant allows a single client per folder — close the previous one when re-running this cell
if "client" in globals():
    client.close()

print("Qdrant path:", QDRANT_PATH)
client = qdrant_client.QdrantClient(path=QDRANT_PATH)

# Start from a clean collection so it always matches the current video and chunking settings
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=len(chunk_embeddings[0]),        # 1536 dimensions for text-embedding-3-small
        distance=models.Distance.COSINE,      # cosine similarity for semantic search
    ),
)

# One point per chunk: the embedding as the vector, the chunk itself (text + position) as the payload
client.upsert(
    collection_name=COLLECTION_NAME,
    points=[
        models.PointStruct(id=chunk["id"], vector=embedding, payload=chunk)
        for chunk, embedding in zip(chunks, chunk_embeddings)
    ],
)

print(f"Stored {client.count(COLLECTION_NAME).count} points in collection '{COLLECTION_NAME}'")

Qdrant path: c:\_home\repos\rag-implementation-fde\qdrant_data
Stored 9 points in collection 'youtube_transcript'


## 5. Accept a User Query and Retrieve the Most Relevant Chunks

This is the semantic search step. `retrieve_chunks()` embeds the user query with the same model used for the chunks, then asks Qdrant for the `TOP_K` chunks whose vectors are closest to it.

Because the comparison is between meanings rather than keywords, a question can match a chunk even when it uses different words. Each retrieved chunk comes back with its similarity `score` (higher is more relevant).

In [9]:
# Retrieval controls
TOP_K = 3                  # number of chunks retrieved for each query


def retrieve_chunks(user_query: str, top_k: int = TOP_K) -> list:
    """
    Retrieves the transcript chunks most relevant to the user query using semantic search.

    The query is embedded with the same model as the chunks, then Qdrant returns
    the `top_k` chunks whose vectors are closest to it (cosine similarity).

    Args:
        user_query (str): The user's question.
        top_k (int): How many chunks to retrieve.

    Returns:
        list[dict]: The retrieved chunks, most relevant first. Each one is the stored chunk
        (id, text, start_char, end_char) plus its similarity `score`.
    """
    # Step 1: Convert the user query into a dense vector (embedding)
    query = get_text_embeddings(user_query)

    # Step 2: Retrieve the top-matching chunks from the Qdrant collection
    text_hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query,
        limit=top_k,
    )

    return [{**point.payload, "score": point.score} for point in text_hits.points]

In [10]:
# The question to ask about the video — change it to try other queries
user_query = "What is the difference between MLOps and LLMOps?"

# retrieved_chunks is the context passed down the pipeline (answer generation)
retrieved_chunks = retrieve_chunks(user_query)

print(f"Query: {user_query}\n")
for chunk in retrieved_chunks:
    print(f"[chunk {chunk['id']}] score: {chunk['score']:.3f}")
    print(chunk["text"][:200] + " ...\n")

Query: What is the difference between MLOps and LLMOps?

[chunk 6] score: 0.681
the video, I will present LLM ops which is as mentioned in the beginning of the video is a sub field of MLOps but oriented to the specific field of LLMs. As it's a sub field of MLOps, it inherits the  ...

[chunk 0] score: 0.680
Hi everyone, I hope you're doing well. In this video, I will show you the difference between DevOps, MLOps, and LLMOPS. So to give you an idea about what each term means, for DevOps, it's actually the ...

[chunk 5] score: 0.600
del training and this is used to trace models characteristics and analyze better their performance. Last but not least, we have what we call an orchestrator which as its name indicates automate the se ...



## 6. Generate an Answer Using the Retrieved Context

This is the generation step. `rag_formatted_response()` takes the user query and the retrieved chunks, numbers the chunks, and builds a prompt asking the language model to answer using only that context.

The model is instructed to cite the chunks it used as [1], [2], ... and to say so when the video does not cover the question, which keeps the answer grounded in the transcript instead of the model's general knowledge.

In [11]:
# OpenAI chat model used to generate the final answer
GENERATION_MODEL = "gpt-5.6-luna"


def rag_formatted_response(user_query: str, context: list):
    """
    Generate a response to the user query using the provided context,
    with chunk references formatted as [1][2], etc.

    This function performs the final step in the RAG pipeline—synthesizing an answer
    from the retrieved transcript chunks (context). It prompts the model to generate a
    grounded response, explicitly citing the chunks it used.

    Args:
        user_query (str): The user's original question.
        context (list): The chunks retrieved from Qdrant (as returned by retrieve_chunks).

    Returns:
        str: A generated response grounded in the retrieved context, with numbered citations.
    """

    # Number the retrieved chunks so the model can cite them as [1], [2], ...
    formatted_context = "\n\n".join(
        f"[{i}] {chunk['text']}" for i, chunk in enumerate(context, start=1)
    )

    # Construct a RAG prompt that includes both:
    # 1. The user's query
    # 2. The supporting transcript chunks
    # The prompt instructs the model to answer using only the provided context,
    # and to include citations like [1], [2], etc. based on the chunk order.
    rag_prompt = f"""
       You answer questions about a YouTube video titled "{video_title}".
       Based only on the given context (excerpts from the video transcript), answer the user query: {user_query}\nContext:\n{formatted_context}\n
       Employ references to the ID of the excerpts provided [ID], ensuring their relevance to the query.
       The referencing should always be in the format of [1][2]... etc.
       If the context does not contain the answer, say that the video does not cover it.
    """

    # Call the chat model to generate the response using the RAG-style prompt
    response = openaiclient.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content": rag_prompt},
        ]
    )

    # Return the model's generated answer
    return response.choices[0].message.content

In [12]:
# Answer the query from section 5 using the chunks retrieved for it
answer = rag_formatted_response(user_query, retrieved_chunks)

print(f"Query: {user_query}\n")
print(answer)

Query: What is the difference between MLOps and LLMOps?

**MLOps** applies DevOps practices to the broader machine-learning lifecycle. It covers automation of training and deployment, versioning of code, data, and models, experiment tracking, testing, monitoring, and reproducibility [2][3].

**LLMOps** is a specialized subset of MLOps focused specifically on large language models. It inherits MLOps and DevOps principles but adds LLM-specific practices such as prompt versioning, testing and monitoring, guardrails for inputs and outputs, feedback loops for improvement, and cost management for computationally expensive models [1][3].


## 7. Putting It All Together

`youtube_rag()` runs the whole query-time pipeline in one call: embed the question, retrieve the top chunks from Qdrant, and generate a grounded answer. It also prints which chunks were retrieved and their similarity scores, so each citation can be traced back to the transcript.

The last question below is deliberately unrelated to the video, to check that the system declines instead of answering from general knowledge.

In [13]:
def youtube_rag(user_query: str, top_k: int = TOP_K):
    """
    Runs the full RAG pipeline for one question: retrieve the most relevant
    transcript chunks, then generate an answer grounded in them.

    Args:
        user_query (str): The user's question about the video.
        top_k (int): How many chunks to retrieve as context.

    Returns:
        str: The generated answer.
    """
    # Terminal color codes to make the printed output easier to read
    CYAN = "\033[96m"
    GREY = "\033[90m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

    # Step 1: Retrieve the most relevant chunks
    context = retrieve_chunks(user_query, top_k)
    sources = ", ".join(f"[{i}] chunk {chunk['id']} ({chunk['score']:.3f})"
                        for i, chunk in enumerate(context, start=1))
    print(f"{GREY}📍 Retrieved: {sources}{RESET}\n")

    # Step 2: Generate the answer from the retrieved context
    answer = rag_formatted_response(user_query, context)
    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{answer}\n")
    return answer

In [14]:
answer = youtube_rag("What is the difference between MLOps and LLMOps?")

👤 User Query: What is the difference between MLOps and LLMOps?

📍 Retrieved: [1] chunk 6 (0.681), [2] chunk 0 (0.680), [3] chunk 5 (0.600)

🤖 BOT RESPONSE:

**MLOps** applies DevOps practices to machine-learning systems broadly. It covers the end-to-end ML lifecycle, including versioning code, data, and models; experiment tracking; testing; monitoring model performance; reproducibility; and automating retraining and deployment.[3]

**LLMOps** is a specialized **subfield of MLOps** focused specifically on large language models. It inherits MLOps and DevOps principles but adds concerns such as:

- Versioning, testing, and monitoring prompts  
- Guardrails to control harmful outputs, toxic behavior, and private-information leaks  
- User-feedback loops for continual improvement and fine-tuning  
- Managing the often high computational cost of LLMs[1][3]

In short: **MLOps manages machine-learning systems in general, while LLMOps applies those practices specifically to LLM-based systems an

In [15]:
answer = youtube_rag("Why is prompt management important?")

👤 User Query: Why is prompt management important?

📍 Retrieved: [1] chunk 8 (0.326), [2] chunk 6 (0.281), [3] chunk 7 (0.272)

🤖 BOT RESPONSE:

Prompt management is important in LLMOps because prompts directly affect the model’s responses and performance. It involves **versioning, testing, and monitoring prompts** so teams can identify which prompts work best and track changes over time [2]. Monitoring also helps track the prompts used, generated responses, model performance, and overall costs [1]. This supports continuous improvement, reliable deployments, and better cost control.



In [16]:
answer = youtube_rag("What is the capital of France?")

👤 User Query: What is the capital of France?

📍 Retrieved: [1] chunk 1 (0.058), [2] chunk 2 (0.050), [3] chunk 0 (0.033)

🤖 BOT RESPONSE:

The video does not cover the capital of France.

